In [ ]:
# Core libraries
import os
import glob
from typing import Union, List
from datetime import datetime, timedelta, time

# Data handling
import pandas as pd
import numpy as np
from scipy import stats

# Plotting
import matplotlib.pyplot as plt
from matplotlib.pyplot import text
from cycler import cycler

# Set global plot font to Times New Roman
csfont = {'fontname': 'Times'}

In [ ]:
# === Time Configuration ===
PRECISION_MINUTES = 1  # Time precision in minutes

# Define the time window of interest: from 10:00 AM to 4:00 PM
START_HOUR = 10
END_HOUR = 16

# Time constants
MINUTES_PER_HOUR = 60
SECONDS_PER_HOUR = 3600
SECONDS_IN_DAY = 24 * 3600

# Compute start and end indices in units of PRECISION_MINUTES
time_focus_start = START_HOUR * MINUTES_PER_HOUR // PRECISION_MINUTES
time_focus_end = END_HOUR * MINUTES_PER_HOUR // PRECISION_MINUTES

# Total number of time intervals in the defined window
num_time_intervals = time_focus_end - time_focus_start

# === Data Path ===
# Path to the folder containing CSV files
data_folder_path = '/Users/ebrukasikaralar/data_parallel_server/data_july02_oct02/csv_files/'

# Calculating Average Number of Agents

In [ ]:
def get_file_list(folder_path: str, file_pattern: str) -> List[str]:
    """
    Returns a list of file paths from the specified folder that match a given pattern.

    Parameters:
    - folder_path (str): Path to the folder containing the files.
    - file_pattern (str): Pattern to match filenames (e.g., '*.csv').

    Returns:
    - List[str]: List of matching file paths.

    Raises:
    - FileNotFoundError: If the specified folder does not exist.
    """

    if not os.path.exists(folder_path):
        raise FileNotFoundError(f"Folder not found: {folder_path}")

    file_list = glob.glob(os.path.join(folder_path, file_pattern))

    if not file_list:
        print(f"[Warning] No files found in '{folder_path}' matching pattern '{file_pattern}'.")

    return file_list

In [ ]:
# Retrieve and sort file lists for agent records and agent events
file_list_agent_records = np.sort(get_file_list(data_folder_path, file_pattern="*_agent_records.csv"))
file_list_agent_events = np.sort(get_file_list(data_folder_path, file_pattern="*_agent_events.csv"))

In [ ]:
# === Identify unique agent groups (excluding main_service == 0) ===
unique_agent_groups = set()

for file in file_list_agent_events:
    df = pd.read_csv(file)
    valid_rows = df[df["main_service"] != 0]
    unique_agent_groups.update(valid_rows["agent_group"].unique())

sorted_agent_groups = np.sort(list(unique_agent_groups))

# === Collect unique main_service and service values for each agent group ===
unique_main_services = {group: set() for group in sorted_agent_groups}
unique_services = {group: set() for group in sorted_agent_groups}

for file in file_list_agent_events:
    df = pd.read_csv(file)

    for group in sorted_agent_groups:
        group_rows = df[df["agent_group"] == group]
        unique_main_services[group].update(group_rows["main_service"].unique())
        unique_services[group].update(group_rows["service"].unique())

# Remove zero entries from service values
for group in sorted_agent_groups:
    unique_services[group].discard(0)

# === Display results ===
for group in sorted_agent_groups:
    main_s = sorted(unique_main_services[group])
    serv_s = sorted(unique_services[group])
    print(f"Agent Group: {group}, Main Services: {main_s}, Services: {serv_s}")

In [ ]:
# === Check whether agent group codes change across different days ===
unique_agent_groups = set()

for file in file_list_agent_events:
    df = pd.read_csv(file)
    valid_rows = df[df["main_service"] != 0]  # Exclude calls that didn't reach offered volume
    agent_groups = np.sort(valid_rows["agent_group"].unique())
    unique_agent_groups.update(agent_groups)

    print(f"{os.path.basename(file)} → Agent Groups: {agent_groups}")

In [ ]:
def reader(file_path: str) -> np.ndarray:
    """
    Reads a CSV file and returns the unique record IDs of calls 
    where the customer subcall count is greater than 1.

    Parameters:
    - file_path (str): Path to the CSV file.

    Returns:
    - np.ndarray: Unique record IDs of relevant calls.
    """
    try:
        df = pd.read_csv(file_path)
    except Exception as e:
        raise FileNotFoundError(f"Failed to read file '{file_path}': {e}")

    if "cust_subcall" not in df.columns or "record_id" not in df.columns:
        raise ValueError(f"Missing required columns in file: {file_path}")

    filtered_ids = df[df["cust_subcall"] > 1]["record_id"].unique()
    return filtered_ids

In [ ]:
# Collecting unique IDs from all files that have more than 1 subcall
data_sets = [reader(file) for file in file_list_agent_records]

In [ ]:
def seconds(array: Union[List[timedelta], pd.Series]) -> np.ndarray:
    """
    Converts an array-like of timedelta objects to an array of seconds.

    Parameters:
    - array (Union[List[timedelta], pd.Series]): An array-like structure of timedelta objects.

    Returns:
    - np.ndarray: An array of seconds corresponding to the timedelta objects.
    """
    if not all(isinstance(td, timedelta) for td in array):
        raise ValueError("All elements of the array must be timedelta objects")

    if isinstance(array, pd.Series):
        # Vectorized operation for pandas Series
        return array.dt.seconds.to_numpy()
    else:
        # List comprehension for a list
        return np.array([td.seconds for td in array])

In [ ]:
def counter(data, precision, num):
    """
    Computes the average number of available agents over the day, adjusting for
    sign-in/out events, breaks, and second subcalls.

    Parameters:
    - data (pd.DataFrame): Event log data.
    - precision (int): Number of minutes per time bucket.
    - num (int): Index for accessing precomputed second-subcall record IDs.

    Returns:
    - np.ndarray: Mean agent availability per interval.
    """
    
    # Sign-in times
    data_sign_in = data[((data["event_id"] == 20) | (data["event_id"] == 21))] #events 20 and 21 are sign-in times
    data_sign_in = data_sign_in.sort_values(["agent","event_start"])
    data_sign_in = data_sign_in.reset_index()
    sign_in = pd.to_timedelta(data_sign_in["event_start"], unit = 'seconds')
    sign_in_times = seconds(sign_in) # Converting sign_in times to seconds
    
    # Sign_out times
    data_sign_out = data[((data["event_id"] == 30) | (data["event_id"] == 31))] #events 30 and 31 are sign-out times
    data_sign_out = data_sign_out.sort_values(["agent","event_start"])
    data_sign_out = data_sign_out.reset_index()
    sign_out = pd.to_timedelta(data_sign_out["event_start"], unit = 'seconds')
    sign_out_times = seconds(sign_out) # Converting sign_out times to seconds
    
    available = np.zeros(SECONDS_IN_DAY) # The number of seconds in a day
    for i in range(len(data_sign_in)):
        if sign_in_times[i] > sign_out_times[i]:
            sign_out_times[i] = SECONDS_IN_DAY - 1
        for j in range(int(sign_in_times[i]),int(sign_out_times[i])+1):
            available[int(j)] += 1
    
    # Breaks
    data_break = data[((data["event_id"] == 60) | (data["event_id"] == 61) | (data["event_id"] == 62))] #events 60, 61,62 are break times
    data_break = data_break.sort_values(["agent","event_start"])
    data_break = data_break.reset_index()
    
    break_start = pd.to_timedelta(data_break["event_start"], unit = 'seconds')
    break_end = pd.to_timedelta(data_break["event_end"], unit = 'seconds')
    
    break_start = seconds(break_start)
    break_end = seconds(break_end)
    
    for i in range(len(break_start)):
        for j in range(int(break_start[i]),int(break_end[i]) + 1):
            available[int(j)] -= 1
        
    # Second subcalls eliminated
    data_ss = data[(data["record_id"].isin(data_sets[num]))]
    data_ss = data_ss.sort_values(["agent", "event_start"])
    data_ss = data_ss.reset_index()
    
    ss_start = pd.to_timedelta(data_ss["event_start"], unit = "seconds")
    ss_end = pd.to_timedelta(data_ss["event_end"], unit = "seconds")
    
    ss_start = seconds(ss_start)
    ss_end = seconds(ss_end)
    
    for k in range(len(ss_start)):
        for l in range(int(ss_start[k]), int(ss_end[k]) + 1):
            available[int(l)] -= 1
        
    available = np.reshape(available, (-1,precision*60))
        
    mean = np.mean(available, axis = 1) 
    variance = np.var(available, axis = 1)   
    
    return mean

In [ ]:
def filt_agents(file_path: str, precision: int, agent_group_indices: list, num: int) -> np.ndarray:
    """
    Computes the average availability of agents in specified agent groups
    over the course of a day, segmented by the given time precision.

    Parameters:
    - file_path (str): Path to the CSV file containing event data.
    - precision (int): Time resolution in minutes for aggregation (e.g., 1 = per minute).
    - agent_group_indices (list): List of agent group IDs to analyze.
    - num (int): Index for selecting the relevant second-subcall data set.

    Returns:
    - np.ndarray: 2D array of mean availability for each agent group (rows) 
                  and time interval (columns).
    """
    try:
        df = pd.read_csv(file_path)
    except Exception as e:
        raise FileNotFoundError(f"Failed to read file '{file_path}': {e}")

    # Define switching agent groups explicitly
    group_switch_map = {
        15: [15, 16],
        19: [19, 20]
    }

    mean_avail_list = []

    for group in agent_group_indices:
        groups_to_include = group_switch_map.get(group, [group])
        group_data = df[df["agent_group"].isin(groups_to_include)]

        mean_avail = counter(group_data, precision, num)
        mean_avail_list.append(mean_avail)

    return np.array(mean_avail_list)

In [ ]:
def calculate_average_agent_availability(file_list: list, agent_group_indices: list, precision: int):
    """
    Calculates the average agent availability across multiple days (CSV files),
    along with daily total availability per time interval.

    Parameters:
    - file_list (list): List of file paths (one per day).
    - agent_group_indices (list): List of agent group IDs to include.
    - precision (int): Time resolution in minutes.

    Returns:
    - average_mean (np.ndarray): Mean availability per agent group and time interval.
    - daily_totals (list of np.ndarray): List of daily total availability across all agent groups.
    """
    total_availability = None
    daily_totals = []

    for day_idx, file_path in enumerate(file_list):
        print(f"Processing file {day_idx + 1}/{len(file_list)}: {file_path}")

        # Get availability for each agent group for the current file
        daily_availability = filt_agents(file_path, precision, agent_group_indices, day_idx)

        # Sum across agent groups to get total availability per time interval
        daily_total = np.sum(daily_availability, axis=0)
        daily_totals.append(daily_total)

        # Initialize accumulator on first pass
        if total_availability is None:
            total_availability = np.zeros_like(daily_availability)

        total_availability += daily_availability

    # Compute the average availability across all days
    average_mean = total_availability / len(file_list)

    return average_mean, daily_totals

In [ ]:
# === Agent Groups of Interest ===
sorted_agent_groups = np.array([1, 5, 9, 15, 19, 26, 28, 30, 31, 33, 34, 38, 39, 40])

# === Compute Mean Availability ===
agents_mean, daily_totals = calculate_average_agent_availability(
    file_list_agent_events, sorted_agent_groups, PRECISION_MINUTES
)

# === Focus on the 10 AM to 4 PM interval ===
interval_length = time_focus_end - time_focus_start
x_axis = np.arange(1, interval_length + 1)  # 1-minute resolution

# Subset the time window of interest
agents_during_focus = agents_mean[:, time_focus_start:time_focus_end]

# === Compute Area Under the Availability Curve ===
total_area = np.trapz(agents_during_focus, x_axis)

# === Estimate Average Number of Agents Needed ===
average_num_agents = np.ceil(total_area / interval_length)

In [ ]:
average_num_agents

# Calculating the percentage of calls handled by each agent group

In [ ]:
def filtered(file_path, service, group, node=0):
    """
    Filter data based on service and node, removing outliers and abnormal outcomes.

    Parameters:
    - file_path: str - Path to the data file.
    - service: int - Service type to filter.
    - node: int - Node number to filter, default is 0.

    Returns:
    - Series: Work time data after filtering.
    """
    
    try:
        df = pd.read_csv(file_path)
    
    except Exception as e:
        raise FileNotFoundError(f"Error reading file {file_path}: {e}")
    
    # Merge agent groups with changing codes over time
    group_map = {15: [15, 16], 19: [19, 20]}
    relevant_groups = group_map.get(group, [group])
    df = df[df["agent_group"].isin(relevant_groups)]

    # Remove calls with abnormal outcomes
    abnormal_outcomes = [4, 13, 14, 23, 30, 40, 50]
    abnormal_ids = df[df["outcome"].isin(abnormal_outcomes)]["call_id"].unique()
    df = df[~df["call_id"].isin(abnormal_ids)]

    # Keep only first subcalls
    df = df[df["cust_subcall"] == 1]
    
    # Transfer calls that happen between two serving parties 
    transfer_calls = df[df["seg_type"].isin([7,8,9])] 
    matching_rows = df['call_id'].isin(transfer_calls['call_id'])
    data_matching = df[matching_rows]
    
    # Calculate the sum of work time for each call id
    work_time_sum = data_matching.groupby(["call_id", "agent"])["work_time"].sum()
    
    first_observations = data_matching.drop_duplicates(subset = ['call_id',"agent"], keep='first')
    
    # Set the work time for these first observations
    for (call_id, agent), total_work_time in work_time_sum.items():
        condition = (first_observations['call_id'] == call_id) & (first_observations["agent"] == agent)
        first_observations.loc[condition, 'work_time'] = total_work_time

    df = pd.concat([df[~matching_rows], first_observations])
    
    # Remove outliers based on queue and work time
    df = df[(df["queue_time"] < 900) & (df["work_time"] < 1800)]

    # Filter by service
    df = df[df["service"] == service]

    # For Retail service (1), filter by node
    if service == 1:
        df = df[df["node"] == node]

    # Filter to time window: 10:00 AM to 4:00 PM
    df["start_time"] = pd.to_datetime(df["start_time"], format="%m/%d/%y %H:%M:%S")
    df["end_time"] = pd.to_datetime(df["end_time"], format="%m/%d/%y %H:%M:%S")
    df = df[(df["start_time"].dt.hour >= START_HOUR) & (df["end_time"].dt.hour < END_HOUR)]

    return len(df)

In [ ]:
def sum_obs(file_list: list, service: int, group: int, node: int) -> int:
    """
    Computes the total number of filtered call records across multiple days.

    Parameters:
    - file_list (list of str): List of file paths (one per day).
    - service (int): Service ID to filter by.
    - group (int): Agent group ID to filter by.
    - node (int): Node number to filter by (used when service == 1).

    Returns:
    - int: Total number of call records after filtering across all files.
    """
    total_calls = sum(filtered(file, service, group, node) for file in file_list)
    return total_calls

In [ ]:
# === Setup: Class Names and Service Mapping ===
all_class_names = [
    "Retail_Node1", "Retail_Node2", "Retail_Node3", "Premier", "Business", "Platinum",
    "Consumer_Loans", "Online_Banking", "EBO", "Telesales", "Subanco", "Case_Quality", "Priority_Service"
]

# Mapping of each class to a service ID
service_class_indices = [1, 1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]

# Agent groups of interest
group_indices = sorted_agent_groups

# === Initialize Results DataFrame ===
num_obs_df = pd.DataFrame(index=all_class_names, columns=group_indices)

# === Populate Observations Table ===
for group_id in group_indices:
    print("group_id: ", group_id)
    for class_idx, service_id in enumerate(service_class_indices):
        print("class_idx: ", class_idx)
        # Determine node number for Retail classes
        node = class_idx + 1 if class_idx < 3 else 0
        
        # Compute number of observations for this group-service-node combination
        num_obs = sum_obs(file_list_agent_records, service_id, group_id, node)
        
        # Fill DataFrame
        num_obs_df.loc[all_class_names[class_idx], group_id] = num_obs

# Optional: convert DataFrame values to integer type if desired
num_obs_df = num_obs_df.astype(int)

# Display final DataFrame
print(num_obs_df)

In [ ]:
# Compute column-wise total, replacing zeros to avoid division by zero
total = num_obs_df.sum(axis=0).replace(0, np.nan)

# Calculate percentages
percentages_all = (num_obs_df.divide(total, axis=1) * 100).fillna(0)

# Set very small values to 0
percentages_no_elimination = percentages_all.copy()
percentages_no_elimination[percentages_no_elimination < 0.1] = 0

In [ ]:
np.round(percentages_all,2)

In [ ]:
num_obs_df_trans = num_obs_df.T
percentages_no_elimination_trans = percentages_no_elimination.T

In [ ]:
# Assignments after elimination based on percentages
filtered_df = num_obs_df_trans[percentages_no_elimination_trans > 0].fillna(0)
filtered_df

In [ ]:
# Assignments after merging the service stations
group_mappings = {
    1: [1],
    2: [5],
    3: [9],
    4: [26,28,30],
    5: [19,20],
    6: [31],
    7: [33],
    8: [34],
    9: [15,16,38,39,40]
}

# Create a DataFrame to store the summed results
grouped_sums = pd.DataFrame(index=group_mappings.keys(), columns=filtered_df.columns)

# Iterate over each group and sum the relevant rows in filtered_df
for group, indices in group_mappings.items():
    grouped_sums.loc[group] = filtered_df.loc[filtered_df.index.intersection(indices)].sum()

# Convert to numeric in case of type mismatch
grouped_sums = grouped_sums.fillna(0)
grouped_sums 

# Calculating the $\mu_{kj}$ for each $(k,j) \in \mathcal{E}$

In [ ]:
# Service times -- For each MERGED GROUP
def filt_service(file_path, service, group, node=0):
    """
    Filter data based on service and node, removing outliers and abnormal outcomes.

    Parameters:
    - file_path: str - Path to the data file.
    - service: int - Service type to filter.
    - node: int - Node number to filter, default is 0.

    Returns:
    - Series: Work time data after filtering.
    """
    
    try:
        data = pd.read_csv(file_path)
    except Exception as e:
        raise FileNotFoundError(f"Error reading file {file_path}: {e}")
    
    group_mapping = {
        1: [1],
        2: [5],
        3: [9],
        4: [26, 28, 30],
        5: [19, 20],
        6: [31],
        7: [33],
        8: [34],
        9: [15, 16, 38, 39, 40]
    }

    # Apply the relevant group filter
    if group not in group_mapping:
        raise ValueError(f"Invalid group identifier: {group}")
        
    data = data[data["agent_group"].isin(group_mapping[group])]     
        
    # eliminating abnormal outcomes
    abnormal_outcomes = [4,13,14,23,30,40,50]
    calls_with_abnormal_outcome = data[data["outcome"].isin(abnormal_outcomes)]
    call_ids_abnormal_outcome = calls_with_abnormal_outcome["call_id"].unique()
    drop_index = data[data["call_id"].isin(call_ids_abnormal_outcome)].index
    data = data.drop(index = drop_index)
        
    # focusing only on the 1st customer subcalls
    data = data[data["cust_subcall"] == 1]
    
    # transfer calls that happen between two serving parties 
    transfer_calls = data[data["seg_type"].isin([7,8,9])] 
    matching_rows = data['call_id'].isin(transfer_calls['call_id'])
    data_matching = data[matching_rows]
    
    # calculate the sum of work time for each call id
    work_time_sum = data_matching.groupby(["call_id", "agent"])["work_time"].sum()
    
    first_observations = data_matching.drop_duplicates(subset = ['call_id',"agent"], keep='first')
    
    # set the work time for these first observations
    for (call_id, agent), total_work_time in work_time_sum.items():
        condition = (first_observations['call_id'] == call_id) & (first_observations["agent"] == agent)
        first_observations.loc[condition, 'work_time'] = total_work_time

    data = pd.concat([data[~matching_rows], first_observations])
    
    data = data[data["queue_time"] < 900] # eliminating outliers through distribution plot of queueing time
    data = data[data["work_time"] < 1800] # eliminating outliers through distribution plot of work time
    
    data = data[data["service"] == service]    

    # if service is 1, we focus on splitting to different nodes for Retail class
    if service == 1:
        
        data = data[data["node"] == node]
    
    # Filter to keep only data where start_time > 10:00 and start_time < 16:00
    data["start_time"] = pd.to_datetime(data["start_time"], format="%m/%d/%y %H:%M:%S")
    data["end_time"] = pd.to_datetime(data["end_time"], format="%m/%d/%y %H:%M:%S")
    data = data[(data["start_time"].dt.hour >= START_HOUR) & (data["end_time"].dt.hour < END_HOUR)]
   
    return data["work_time"] # in the database worktime is calculated in seconds

In [ ]:
def means_service(file_list, service, group, node):
    """
    Calculates the average service time across multiple days for a given service and node.

    Parameters:
    - file_list: list of str - List of file paths.
    - service: int - Service type to filter.
    - node: int - Node number to filter.

    Returns:
    - float: Average service time.
    """

    main_dataframe = pd.DataFrame(filt_service(file_list[0],service, group, node))
    
    for i in range(1,len(file_list)):
        
        print(i)
        data = filt_service(file_list[i],service,group,node)
        df = pd.DataFrame(data)
        main_dataframe = pd.concat([main_dataframe,df],axis=0)
        
    if main_dataframe.empty:
        
        return 0  # Return 0 or appropriate value if no data is available

    return main_dataframe.mean()[0]

In [ ]:
all_class_names = ["Retail_Node1", "Retail_Node2", "Retail_Node3", "Premier", "Business", "Platinum", "Consumer_Loans", "Online_Banking", "EBO", "Telesales", "Subanco", "Case_Quality", "Priority_Service"]

group_indices = [1, 2, 3, 4, 5, 6, 7, 8, 9]

service_times_df = pd.DataFrame(columns = group_indices, index = all_class_names)

service_class_indices = [1, 1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11] ## Classes served during this period


for ind in group_indices: 
    
    for i, service in enumerate(service_class_indices):
                
        node = i + 1 if i < 3 else 0
        
        service_time = means_service(file_list_agent_records, service, ind, node)
        
        service_times_df.loc[all_class_names[i], ind] = service_time
        
        print(service_times_df)

In [ ]:
service_times_df = service_times_df.T
filtered_service_times = service_times_df.where(grouped_sums > 0, 0)
filtered_service_times

In [ ]:
service_rates_df = filtered_service_times.applymap(lambda x: 3600 / x if x != 0 else 0)
service_rates_df